In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName('test') \
    .getOrCreate()

Question 1

In [2]:
spark.version

'3.5.0'

Question 2

In [3]:
df_temp = spark.read.parquet("../data/raw/*.parquet")

In [5]:
df_temp = df_temp.repartition(4)
df_temp.write.parquet('../data/pt/2025/11/')
del df_temp

In [6]:
from pathlib import Path

output_dir = Path("../data/pt/2025/11/")

# Only count actual parquet data files
files = list(output_dir.glob("*.parquet"))

sizes_bytes = [f.stat().st_size for f in files]
avg_size_bytes = sum(sizes_bytes) / len(sizes_bytes) if sizes_bytes else 0

print(f"Number of parquet files: {len(files)}")
print(f"Average size: {avg_size_bytes:.2f} bytes")
print(f"Average size: {avg_size_bytes / 1024 / 1024:.2f} MB")

Number of parquet files: 4
Average size: 26556660.75 bytes
Average size: 25.33 MB


Question 3

In [11]:
df = spark.read.parquet('../data/pt/2025/11/')

df.createOrReplaceTempView('trips_data')

In [29]:
spark.sql("DESCRIBE trips_data").show(truncate=False)

+---------------------+-------------+-------+
|col_name             |data_type    |comment|
+---------------------+-------------+-------+
|VendorID             |int          |NULL   |
|tpep_pickup_datetime |timestamp_ntz|NULL   |
|tpep_dropoff_datetime|timestamp_ntz|NULL   |
|passenger_count      |bigint       |NULL   |
|trip_distance        |double       |NULL   |
|RatecodeID           |bigint       |NULL   |
|store_and_fwd_flag   |string       |NULL   |
|PULocationID         |int          |NULL   |
|DOLocationID         |int          |NULL   |
|payment_type         |bigint       |NULL   |
|fare_amount          |double       |NULL   |
|extra                |double       |NULL   |
|mta_tax              |double       |NULL   |
|tip_amount           |double       |NULL   |
|tolls_amount         |double       |NULL   |
|improvement_surcharge|double       |NULL   |
|total_amount         |double       |NULL   |
|congestion_surcharge |double       |NULL   |
|Airport_fee          |double     

In [13]:
sql_query = '''
SELECT COUNT(*)
FROM trips_data
WHERE tpep_pickup_datetime >= '2025-11-15 00:00:00' AND tpep_pickup_datetime < '2025-11-16 00:00:00'
'''
spark.sql(sql_query).show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



Question 4

In [18]:
sql_query = '''
SELECT TIMESTAMPDIFF(HOUR,tpep_pickup_datetime,tpep_dropoff_datetime) AS length_trip
FROM trips_data
ORDER BY length_trip DESC
LIMIT 1
'''
spark.sql(sql_query).show()

+-----------+
|length_trip|
+-----------+
|         90|
+-----------+



Question 5: 4040

Question 6

In [27]:
df_zones = spark.read.option("header", "true").csv("../data/taxi_zone_lookup.csv")
df_zones.createOrReplaceTempView('zones')
df_zones.head(2)

[Row(LocationID='1', Borough='EWR', Zone='Newark Airport', service_zone='EWR'),
 Row(LocationID='2', Borough='Queens', Zone='Jamaica Bay', service_zone='Boro Zone')]

In [28]:
spark.sql("DESCRIBE zones").show(truncate=False)

+------------+---------+-------+
|col_name    |data_type|comment|
+------------+---------+-------+
|LocationID  |string   |NULL   |
|Borough     |string   |NULL   |
|Zone        |string   |NULL   |
|service_zone|string   |NULL   |
+------------+---------+-------+



In [32]:
sql_query = '''
SELECT z.Zone,
    COUNT(*) as cnt
FROM trips_data AS t
LEFT JOIN zones AS z
ON t.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY cnt ASC
LIMIT 5
'''
spark.sql(sql_query).show(truncate=False)

+---------------------------------------------+---+
|Zone                                         |cnt|
+---------------------------------------------+---+
|Eltingville/Annadale/Prince's Bay            |1  |
|Governor's Island/Ellis Island/Liberty Island|1  |
|Arden Heights                                |1  |
|Port Richmond                                |3  |
|Rikers Island                                |4  |
+---------------------------------------------+---+



In [33]:
!jupyter nbconvert --to html questions_answers.ipynb --output homework6_answers.html

[NbConvertApp] Converting notebook questions_answers.ipynb to html
[NbConvertApp] Writing 291499 bytes to homework6_answers.html
